[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week5_nlp_llms/day36_llm_agents/day36_notebook.ipynb)

# Day 36 / 42: LLM Agents
### #42DaysOfML | Week 5: NLP and LLMs

**Resources used to build this notebook:**
- ReAct paper: *ReAct: Synergizing Reasoning and Acting in Language Models* (Yao et al., 2022)
- [HuggingFace smolagents](https://github.com/huggingface/smolagents) — minimal agent framework
- [mlabonne/llm-course](https://github.com/mlabonne/llm-course) — LLM engineer track reference

---

## What You'll Learn
1. What makes an LLM an agent vs a chatbot
2. The ReAct loop: Reason, Act, Observe, repeat
3. Build a tool-using agent from scratch with Python only
4. Implement OpenAI function calling for structured tool use
5. Use HuggingFace `smolagents` to build a real agent with web search
6. Where agents fail and how production teams guard against it

---

In [ ]:
!pip install smolagents openai duckduckgo-search wikipedia matplotlib numpy --quiet

## The Concept

A chatbot takes a message and returns a response. That's one step.

An agent takes a goal and runs as many steps as it needs to complete it. At each step it decides: do I have enough information to answer, or do I need to take an action first? If it needs an action, it picks a tool, runs it, reads the result, and decides again.

This loop is called **ReAct**: Reasoning and Acting interleaved. Proposed by Yao et al. in 2022, it showed that letting models write out their reasoning before taking each action dramatically improves accuracy on multi-step tasks.

The pattern is:
```
Thought: [LLM reasons about what to do next]
Action:  [LLM picks a tool and specifies the input]
Observation: [Tool runs, returns a result]
... repeat until ...
Final Answer: [LLM has enough information to answer]
```

Perplexity AI's architecture is this loop. Every query triggers a search action, the result becomes an observation, and the LLM generates a grounded answer from it. The user sees a chatbot. Underneath, it's an agent running two or three tool calls per query.

**Three things every agent needs:**
1. An LLM that can reason and follow the ReAct format
2. A set of tools with clear descriptions the LLM reads to decide which to pick
3. A loop that runs until the LLM signals a final answer or hits a step limit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')

boxes = [
    (1.5, 2.5, "User Goal",          '#BBDEFB', '#1565C0'),
    (3.8, 2.5, "Thought\n(LLM reasons)",  '#C8E6C9', '#2E7D32'),
    (6.2, 2.5, "Action\n(pick + call tool)", '#FFF9C4', '#F57F17'),
    (8.7, 2.5, "Observation\n(tool result)", '#F8BBD0', '#C62828'),
    (11.2,2.5, "Final\nAnswer",       '#E1BEE7', '#6A1B9A'),
]

for i, (x, y, label, fc, ec) in enumerate(boxes):
    patch = FancyBboxPatch((x-1.1, y-0.75), 2.2, 1.5,
                            boxstyle="round,pad=0.08", facecolor=fc, edgecolor=ec, linewidth=2)
    ax.add_patch(patch)
    ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold', color=ec)
    if i < len(boxes)-1:
        ax.annotate('', xy=(boxes[i+1][0]-1.1, y), xytext=(x+1.1, y),
                   arrowprops=dict(arrowstyle='->', color='#444', lw=2))

ax.annotate('', xy=(3.8, 1.75), xytext=(8.7, 1.75),
           arrowprops=dict(arrowstyle='<-', color='#888', lw=1.5, linestyle='dashed'))
ax.text(6.2, 1.35, 'Loop until done or max_steps reached', ha='center', fontsize=8.5, color='#888', style='italic')

ax.set_title('ReAct Agent Loop: Reason  →  Act  →  Observe  →  Repeat', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("The loop only exits in two ways:")
print("  1. The LLM produces a Final Answer")
print("  2. max_steps is reached (safety limit)")
print("Without a step limit, a poorly prompted agent can loop indefinitely.")

## Section 1: Build a Tool-Using Agent From Scratch

No frameworks. Pure Python. This is how you understand what frameworks like LangChain and smolagents are doing underneath.

In [ ]:
import re
import json
from dataclasses import dataclass
from typing import Callable

# ----------------------------------------------------------------
# Tool definition: name, description, function
# The description is what the LLM reads to decide which tool to pick
# Clear, specific descriptions = better tool selection accuracy
# ----------------------------------------------------------------

@dataclass
class Tool:
    name: str
    description: str
    func: Callable

    def run(self, input_str: str) -> str:
        try:
            return str(self.func(input_str))
        except Exception as e:
            return f"Error running {self.name}: {e}"


# ---- Tool implementations ----

def calculator(expression: str) -> str:
    """
    Safe math evaluator. Only allows numeric operations.
    In production: use a proper math parser, not eval().
    """
    allowed = {'abs': abs, 'round': round, 'min': min, 'max': max, 'pow': pow}
    try:
        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as e:
        return f"Invalid expression: {e}"


# Simulated Wikipedia for offline testing
# In production, use the `wikipedia` library or a search API
WIKI_KB = {
    "transformer": "The Transformer architecture was introduced in 'Attention Is All You Need' by Vaswani et al. in 2017 at Google Brain.",
    "bert": "BERT (Bidirectional Encoder Representations from Transformers) was released by Google in October 2018. It has 110M parameters in its base version.",
    "gpt": "GPT-4 was released by OpenAI in March 2023. It supports a 128K token context window as of late 2023.",
    "react": "ReAct is a prompting framework proposed by Yao et al. in 2022. It interleaves reasoning steps with action steps in LLM agents.",
    "faiss": "FAISS (Facebook AI Similarity Search) was developed by Meta AI. It supports exact and approximate nearest neighbour search across billions of vectors.",
    "rag": "RAG (Retrieval-Augmented Generation) was introduced by Lewis et al. in 2020. It grounds LLM answers in retrieved documents, reducing hallucination.",
    "openai": "OpenAI was founded in December 2015 by Sam Altman, Elon Musk, and others. GPT-4 was released in March 2023.",
    "llama": "LLaMA (Large Language Model Meta AI) was released by Meta. LLaMA 3.1 405B was released in July 2024.",
}

def wikipedia_search(query: str) -> str:
    query_lower = query.lower()
    for key, value in WIKI_KB.items():
        if key in query_lower:
            return value
    return f"No Wikipedia result found for '{query}'. Try a more specific term."


def unit_converter(query: str) -> str:
    """
    Simple unit conversions. Input format: 'value unit1 to unit2'
    Example: '100 km to miles'
    """
    conversions = {
        ("km", "miles"): 0.621371,
        ("miles", "km"): 1.60934,
        ("kg", "lbs"): 2.20462,
        ("lbs", "kg"): 0.453592,
        ("celsius", "fahrenheit"): lambda c: c * 9/5 + 32,
        ("fahrenheit", "celsius"): lambda f: (f - 32) * 5/9,
    }
    parts = query.lower().split()
    try:
        value = float(parts[0])
        from_unit = parts[1]
        to_unit = parts[-1]
        conv = conversions.get((from_unit, to_unit))
        if conv is None:
            return f"Conversion from {from_unit} to {to_unit} not supported."
        result = conv(value) if callable(conv) else value * conv
        return f"{value} {from_unit} = {result:.4f} {to_unit}"
    except Exception:
        return "Format: 'value unit1 to unit2' e.g. '100 km to miles'"


# ---- Register tools ----
TOOLS = [
    Tool(
        name="calculator",
        description="Evaluate math expressions. Handles +, -, *, /, **, abs(), round(). Input: math expression string only.",
        func=calculator
    ),
    Tool(
        name="wikipedia",
        description="Look up factual information about ML, AI, and technology topics. Input: a search query string.",
        func=wikipedia_search
    ),
    Tool(
        name="unit_converter",
        description="Convert between units. Input format: 'value unit1 to unit2'. Examples: '100 km to miles', '37 celsius to fahrenheit'.",
        func=unit_converter
    ),
]

# Test all tools
print("TOOL TESTS")
print("=" * 55)
tests = [
    ("calculator",     "2 ** 10"),
    ("calculator",     "15 * 7 + 42"),
    ("wikipedia",      "transformer architecture"),
    ("wikipedia",      "react prompting framework"),
    ("unit_converter", "100 km to miles"),
    ("unit_converter", "37 celsius to fahrenheit"),
]
for tool_name, inp in tests:
    tool = next(t for t in TOOLS if t.name == tool_name)
    result = tool.run(inp)
    print(f"  {tool_name}({inp!r})")
    print(f"    -> {result[:100]}")

In [ ]:
# ----------------------------------------------------------------
# The ReAct loop from scratch
# Simulates what an LLM-powered agent does, using hardcoded
# reasoning traces so you can understand the pattern without
# needing an API key
# ----------------------------------------------------------------

def parse_action(text: str):
    """
    Parse LLM output to extract tool name and input.
    LLM is expected to output:
      Action: tool_name
      Action Input: <input>
    or
      Final Answer: <answer>
    """
    if "Final Answer:" in text:
        answer = text.split("Final Answer:")[-1].strip()
        return "final", answer
    action_match = re.search(r"Action:\s*(\w+)", text)
    input_match  = re.search(r"Action Input:\s*(.+)", text)
    if action_match and input_match:
        return action_match.group(1).strip(), input_match.group(1).strip()
    return None, None


def run_react_agent(goal: str, simulated_steps: list, tools: list, max_steps: int = 8) -> str:
    """
    Simulate a ReAct agent loop.
    In production, `simulated_steps` is replaced by LLM API calls.
    """
    print(f"Goal: {goal}")
    print("=" * 60)
    tool_map = {t.name: t for t in tools}

    for step_num, llm_output in enumerate(simulated_steps, 1):
        if step_num > max_steps:
            return "Max steps reached. Could not complete the task."

        print(f"\nStep {step_num}:")
        # Print the thought (lines before Action:)
        for line in llm_output.strip().split("\n"):
            print(f"  {line}")

        action_name, action_input = parse_action(llm_output)

        if action_name == "final":
            print(f"\nFinal Answer: {action_input}")
            return action_input

        if action_name and action_name in tool_map:
            observation = tool_map[action_name].run(action_input)
            print(f"  Observation: {observation[:120]}")
        elif action_name:
            print(f"  Error: tool '{action_name}' not found")

    return "Max steps reached."


# ---- Demo 1: Multi-step math + lookup ----
steps_demo1 = [
    """Thought: I need to find when the Transformer architecture was introduced, then compute how many years ago that was from 2024.
Action: wikipedia
Action Input: transformer architecture""",

    """Thought: The Transformer was introduced in 2017. Now I calculate 2024 - 2017.
Action: calculator
Action Input: 2024 - 2017""",

    """Thought: The result is 7. The Transformer architecture is 7 years old as of 2024.
Final Answer: The Transformer architecture was introduced in 2017 by Vaswani et al. at Google Brain. That is 7 years ago as of 2024."""
]

result1 = run_react_agent(
    goal="When was the Transformer architecture introduced, and how many years ago was that from 2024?",
    simulated_steps=steps_demo1,
    tools=TOOLS
)

# ---- Demo 2: Unit conversion + calculation ----
print("\n" + "=" * 60)
steps_demo2 = [
    """Thought: I need to convert 42 km to miles, then calculate the time to cover that distance at 60 mph.
Action: unit_converter
Action Input: 42 km to miles""",

    """Thought: 42 km is 26.0976 miles. At 60 mph, time = distance / speed.
Action: calculator
Action Input: 26.0976 / 60""",

    """Thought: The result is 0.4349 hours. I'll convert to minutes: 0.4349 * 60.
Action: calculator
Action Input: 0.4349 * 60""",

    """Thought: That is approximately 26 minutes.
Final Answer: 42 km equals 26.1 miles. Traveling at 60 mph, it takes about 26 minutes."""
]

result2 = run_react_agent(
    goal="How long does it take to travel 42 km at 60 mph?",
    simulated_steps=steps_demo2,
    tools=TOOLS
)

In [ ]:
# ----------------------------------------------------------------
# Section 2: OpenAI Function Calling
# The structured way to do tool use with the OpenAI API
# The model decides which function to call and generates the arguments
# ----------------------------------------------------------------
import os
import json

# Tool specs in OpenAI's function calling format
tools_spec = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical expression. Returns the numeric result as a string.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A valid Python math expression, e.g. '2**10', '15*7+42'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "wikipedia",
            "description": "Search for factual information about ML and AI topics on Wikipedia.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query, e.g. 'transformer architecture'"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "unit_converter",
            "description": "Convert between units. Supports km/miles, kg/lbs, celsius/fahrenheit.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Conversion request e.g. '100 km to miles'"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

tool_functions = {
    "calculator":     calculator,
    "wikipedia":      wikipedia_search,
    "unit_converter": unit_converter,
}

def run_openai_agent(query: str, max_iterations: int = 5):
    """
    Full OpenAI function-calling agent loop.
    Requires OPENAI_API_KEY environment variable.
    """
    api_key = os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        print("[No API key set. Showing what the loop would do.]")
        print(f"  Query sent: {query}")
        print(f"  Tool specs sent: {[t['function']['name'] for t in tools_spec]}")
        print(f"  Expected: model selects a tool and returns structured JSON arguments")
        return

    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    messages = [{"role": "user", "content": query}]

    print(f"Query: {query}")
    print("=" * 55)

    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages,
            tools=tools_spec,
            tool_choice="auto"
        )
        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            print(f"\nFinal Answer: {message.content}")
            return message.content

        for tool_call in message.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            input_val = list(args.values())[0]

            print(f"  Step {iteration+1}: {name}({input_val!r})")
            result = tool_functions[name](input_val)
            print(f"  Result: {result[:100]}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

    return "Max iterations reached."


# Run the agent
run_openai_agent(
    "When was BERT released? How many years ago was that from 2024? Also, convert 37 degrees Celsius to Fahrenheit."
)

print("\nKey difference from the scratch implementation:")
print("  The model generates tool arguments as structured JSON, not free-form text.")
print("  No regex parsing needed. Arguments are always valid for the function signature.")
print("  This is why function calling is the production standard over ReAct string parsing.")

In [ ]:
# ----------------------------------------------------------------
# Section 3: smolagents by HuggingFace
# The lightest production agent framework available.
# Uses CodeAgent: the LLM writes Python code instead of JSON actions.
# Code execution is more flexible than JSON tool calls.
# ----------------------------------------------------------------
from smolagents import CodeAgent, DuckDuckGoSearchTool, HfApiModel, tool

# Define custom tools with the @tool decorator
# The docstring is what the LLM reads to decide whether to use this tool
@tool
def calculator_tool(expression: str) -> str:
    """Evaluates a mathematical expression and returns the result.
    Use this for any arithmetic: addition, subtraction, multiplication,
    division, exponentiation. Input must be a valid Python math expression.
    Example: '2 ** 10', '15 * 7 + 42', '(100 - 32) * 5/9'
    """
    allowed = {'abs': abs, 'round': round, 'min': min, 'max': max}
    try:
        return str(eval(expression, {"__builtins__": {}}, allowed))
    except Exception as e:
        return f"Error: {e}"


@tool
def unit_converter_tool(query: str) -> str:
    """Converts between common units of measurement.
    Supported conversions: km/miles, kg/lbs, celsius/fahrenheit.
    Input format: 'value unit1 to unit2'
    Examples: '100 km to miles', '70 kg to lbs', '100 celsius to fahrenheit'
    """
    conversions = {
        ("km", "miles"): 0.621371, ("miles", "km"): 1.60934,
        ("kg", "lbs"): 2.20462,   ("lbs", "kg"): 0.453592,
        ("celsius", "fahrenheit"): lambda c: c * 9/5 + 32,
        ("fahrenheit", "celsius"): lambda f: (f - 32) * 5/9,
    }
    parts = query.lower().split()
    try:
        value = float(parts[0])
        from_unit, to_unit = parts[1], parts[-1]
        conv = conversions.get((from_unit, to_unit))
        if not conv:
            return f"Conversion {from_unit} to {to_unit} not supported."
        result = conv(value) if callable(conv) else value * conv
        return f"{value} {from_unit} = {result:.4f} {to_unit}"
    except Exception:
        return "Format: 'value unit1 to unit2'"


# Free HuggingFace model — no API key needed if you use HF_TOKEN
# Qwen/Qwen2.5-72B-Instruct is smolagents' recommended open model
model = HfApiModel(model_id="Qwen/Qwen2.5-72B-Instruct")

agent = CodeAgent(
    tools=[DuckDuckGoSearchTool(), calculator_tool, unit_converter_tool],
    model=model,
    max_steps=6,         # safety limit
    verbosity_level=2,   # print each step
)

# Run the agent (requires HF_TOKEN or OPENAI_API_KEY)
queries = [
    "What is the square root of the year BERT was released?",
    "How many kilometers is a marathon (26.2 miles)?",
]

hf_token = os.environ.get("HF_TOKEN", "")
if hf_token:
    for query in queries:
        print(f"\nQuery: {query}")
        print("=" * 50)
        result = agent.run(query)
        print(f"Answer: {result}")
else:
    print("[HF_TOKEN not set. Set it to run the smolagents demo.]")
    print("Get a free token at: huggingface.co/settings/tokens")
    print("\nThe agent would:")
    print("  1. Write Python code to call wikipedia_search or DuckDuckGo for BERT's release year")
    print("  2. Write Python code calling calculator_tool with 'math.sqrt(2018)'")
    print("  3. Return the computed answer")
    print("\nCodeAgent differs from ReAct: it writes executable code, not structured JSON.")
    print("This makes it more flexible but requires a sandboxed execution environment.")

In [ ]:
# ----------------------------------------------------------------
# Section 4: Agent failure modes and how to guard against them
# ----------------------------------------------------------------

failure_modes = [
    'Wrong tool\nselected', 'Infinite\nloop', 'Hallucinated\ntool input',
    'Max steps\nexceeded', 'Correct\nanswer'
]
rates = [18, 12, 22, 8, 40]
colors_f = ['#F44336', '#FF5722', '#FF9800', '#FFC107', '#4CAF50']

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(failure_modes, rates, color=colors_f, alpha=0.85, edgecolor='black')
for bar, r in zip(bars, rates):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{r}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Rate (%)', fontsize=12)
ax.set_title('Agent Outcome Distribution on Complex Queries\n'
             '(Only 40% of runs complete correctly — agents are not plug-and-play)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("\nEach failure mode and its fix:")
print()
print("Wrong tool selected (18%)")
print("  Cause: vague tool descriptions. The model can't tell when to use which tool.")
print("  Fix: write descriptions that include what the tool does NOT handle.")
print("       'Use ONLY for math. Do not use for factual lookups.'")
print()
print("Infinite loop (12%)")
print("  Cause: no max_steps limit. Agent keeps trying the same failing tool.")
print("  Fix: always set max_steps. Log which tool called at which step.")
print("       Add a 'stuck detector': if the last 3 tool calls are identical, stop.")
print()
print("Hallucinated tool input (22%)")
print("  Cause: model generates an argument the tool rejects, agent recovers poorly.")
print("  Fix: validate tool inputs before execution. Return clear error messages.")
print("       'Invalid expression: division by zero' is more recoverable than a crash.")
print()
print("Max steps exceeded (8%)")
print("  Cause: task genuinely requires more steps than the limit allows.")
print("  Fix: tune max_steps per task type. Log all exceeded-step cases for review.")

## Real World Problem: The Infinite Loop in Production

A team at a B2B SaaS company deployed a customer support agent with access to three tools: a ticket lookup, a knowledge base search, and an email sender. They set no step limit.

A user asked: "Can you check if my refund was processed?"

The ticket lookup returned a result, but the agent misread the date format (MM/DD vs DD/MM) and concluded the ticket didn't exist. It called the tool again with a reformatted query. Same result. Same misread. Same call. The agent ran 847 tool calls over 4 minutes before someone noticed the API bill spike.

Three guards that prevent this:

1. **max_steps**: Set a hard limit. 8-10 steps handles 95% of legitimate requests. Anything above that is almost certainly stuck.

2. **Repetition detection**: Track the last 3 tool calls. If the same tool is called with nearly identical inputs twice in a row, stop and return a fallback: "I was unable to complete this request. A human agent will follow up."

3. **Per-tool call limits**: Each tool gets a budget (e.g. max 3 calls per agent run). This prevents one broken tool from consuming the entire step budget.

None of these are complex to implement. All three are skipped more often than not in early agent deployments.

## Interview Corner: MNC-Level Questions

---

**Q1: What is the ReAct framework and why does writing out reasoning before acting improve accuracy?**

*What they're testing:* Whether you understand why the format works, not just that it exists.

*Answer direction:* ReAct interleaves reasoning steps (Thought) with action steps (Action/Observation) instead of jumping straight to a tool call. Writing out the reasoning first forces the model to commit to a plan before acting. Each reasoning step also becomes part of the context for the next generation, so the model conditions on its own intermediate conclusions. This is why it helps most on multi-step tasks: each step builds on the verified output of the previous one, rather than compressing multi-step reasoning into a single token prediction with a high error rate.

---

**Q2: A user asks an agent a question. The agent picks the wrong tool three times in a row. What's the most likely root cause?**

*What they're testing:* Debugging intuition.

*Answer direction:* Almost always the tool descriptions. The model reads descriptions to decide which tool to call. If two tools have similar descriptions, or descriptions don't clearly state what they don't handle, the model will guess incorrectly. Fix: rewrite descriptions to include negative constraints: 'Use this for math expressions only. Do not use for factual questions.' Also check that the tool names themselves are informative. A tool named 'search' is ambiguous. 'Wikipedia factual lookup' is not.

---

**Q3: What is the difference between OpenAI function calling and the ReAct string-parsing approach?**

*What they're testing:* API depth.

*Answer direction:* In ReAct string parsing, the model writes free text that you then parse with regex to extract the tool name and input. This breaks when the model doesn't follow the exact format. Function calling asks the model to return structured JSON matching a schema you define. The model is specifically fine-tuned to produce valid JSON for the function signature. No regex needed. Arguments are always correctly typed and named. It's significantly more reliable in production, which is why it's the standard for tool use over raw ReAct string parsing.

---

**Q4: How does smolagents' CodeAgent differ from a standard ReAct agent?**

*What they're testing:* Awareness of different agent paradigms.

*Answer direction:* A standard ReAct agent generates a tool name and JSON arguments, then waits for the tool result before continuing. CodeAgent, introduced by HuggingFace, generates executable Python code that calls tools as Python functions. The code is executed in a sandbox and the output feeds back into the next step. This has two advantages: the agent can combine multiple tool calls in a single step (call calculator and unit_converter in one code block), and it can use Python logic (conditionals, loops) to handle tool outputs. The tradeoff is that code execution requires a sandboxed environment and is harder to audit than structured JSON actions.

---

**Q5: How would you test an agent before deploying it to production?**

*What they're testing:* Production readiness thinking.

*Answer direction:* Three layers. First, unit test each tool independently with valid inputs, edge case inputs, and deliberately invalid inputs. Verify error messages are clean and recoverable. Second, build a golden set of 50-100 queries with known correct answers. Run the agent on all of them and measure: correct answer rate, average steps to completion, and failure mode distribution (wrong tool, infinite loop, hallucinated input). This is your baseline before any deployment. Third, shadow mode: run the new agent in parallel with the existing system for a week, logging responses without serving them. Compare disagreements. Any case where the agent and the ground truth disagree needs manual review before you increase traffic.

## ML Spotlight

**smolagents by HuggingFace (2024)**

Most agent frameworks (LangChain, AutoGPT, CrewAI) have hundreds of lines of abstraction between your code and the actual LLM call. smolagents was built with the opposite philosophy: the core agent loop is around 1,000 lines of Python. You can read the entire codebase in an afternoon.

It supports two agent types. `ToolCallingAgent` uses function calling style, similar to OpenAI's tool use. `CodeAgent` lets the LLM write and execute Python code directly, which is more flexible for multi-step calculations and data manipulation.

It works with any HuggingFace model via the Inference API, and with OpenAI models. The `@tool` decorator is all you need to expose a Python function as an agent tool.

In their published benchmarks on GAIA (a general AI assistant benchmark), CodeAgent outperforms ReAct-style agents on tasks that require combining more than two tools in a single reasoning step.

GitHub: https://github.com/huggingface/smolagents

Docs: https://huggingface.co/docs/smolagents

## Practice Exercise

**Task 1:** Add a fourth tool to the from-scratch agent: a `word_counter` that takes a string and returns the number of words and characters. Then write a 3-step ReAct trace that uses it.

**Task 2:** Add repetition detection to `run_react_agent`. Track the last 3 (action_name, action_input) pairs. If the same pair appears twice consecutively, stop and return: `"Stuck in a loop. Stopping after {step} steps."`

**Task 3:** Implement a basic agent memory. After each successful run, store the (query, final_answer) pair in a dict. At the start of each new run, check if the new query is semantically similar (using `sentence-transformers` cosine similarity) to a cached query. If similarity is above 0.9, return the cached answer immediately without running the full agent loop.

---

**What's Next**

Day 37: Vector Databases. FAISS is great for prototyping. When your index has 10 million vectors and you need sub-10ms latency with metadata filtering, you need to understand what's happening inside the index.